In [0]:
%run ./00_config

✅ Config loaded : billing.sales_data
📅 Target date   : 2026-09-18


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, countDistinct, round, lit
from pyspark.sql.utils import AnalysisException
from pyspark.sql.utils import AnalysisException

# ============================================================
# GOLD BRAND PERFORMANCE
# ============================================================

SOURCE_TABLE = f"{catalog_name}.{schema_name}.billing_silver"
TARGET_TABLE = f"{catalog_name}.{schema_name}.gold_brand_performance"

print("=" * 70)
print("GOLD BRAND PERFORMANCE")
print("=" * 70)
print(f"Source table : {SOURCE_TABLE}")
print(f"Target table : {TARGET_TABLE}")
print(f"Target date  : {TARGET_DATE}")
print("Excluded     : ['Ho Marlboro']")
print("=" * 70)


# ============================================================
# 1. READ SILVER
# ============================================================

silver = (
    spark.table(SOURCE_TABLE)
    .filter(col("source_date") == TARGET_DATE)
)

silver_count = silver.count()

print(f"📥 Silver rows for {TARGET_DATE}: {silver_count:,}")

if silver_count == 0:
    raise RuntimeError(
        f"❌ No silver data for {TARGET_DATE} — did 02_silver run?"
    )


# ============================================================
# 2. EXCLUDE HO MARLBORO
# ============================================================

EXCLUDED_STORES = [
    "Ho Marlboro",
    "Dummy Store --- For Testing Only"
]

silver = silver.filter(
    ~F.lower(F.trim(F.col("storeName"))).isin(
        [s.lower().strip() for s in EXCLUDED_STORES]
    )
)

filtered_count = silver.count()

print(
    f"📊 Rows after store exclusion: "
    f"{filtered_count:,}"
)


# ============================================================
# 3. BRAND AGGREGATION
# ============================================================
#
# VM aggregate.py logic:
#
# brand_sales:
#   brandName
#   orderDate
#   nooforders = COUNT(DISTINCT invoice)
#   sales      = SUM(totalProductPrice)
#   aov        = sales / distinct invoices
#
# IMPORTANT:
# - No orderStatus filter
# - No categoryName
# - No productId
# - No cost/profit metrics
# - No discount metrics
# - No GST metrics
# - No gross margin
#
# ============================================================

brand_df = (
    silver
    .groupBy(
        col("brandName"),
        col("orderDate")
    )
    .agg(
        countDistinct("invoice").alias("nooforders"),

        round(
            sum("totalProductPrice"),
            2
        ).alias("sales")
    )
    .withColumn(
        "aov",
        round(
            col("sales") / col("nooforders"),
            2
        )
    )
    .withColumn(
        "source_date",
        lit(TARGET_DATE)
    )
)

brand_count = brand_df.count()

print(
    f"📊 Brand aggregate rows: "
    f"{brand_count:,}"
)

if brand_count == 0:
    raise RuntimeError(
        f"❌ Brand aggregation produced 0 rows "
        f"for {TARGET_DATE}"
    )


# ============================================================
# 4. IDEMPOTENT TARGET-DATE REPLACEMENT
# ============================================================

if spark.catalog.tableExists(TARGET_TABLE):

    spark.sql(
        f"""
        DELETE FROM {TARGET_TABLE}
        WHERE source_date = '{TARGET_DATE}'
        """
    )

    print(
        f"🗑️ Cleared existing gold_brand_performance "
        f"for {TARGET_DATE}"
    )

else:

    print(
        "ℹ️ gold_brand_performance does not exist yet "
        "— creating it."
    )


# ============================================================
# 5. WRITE GOLD
# ============================================================

(
    brand_df
    .write
    .mode("append")
    .partitionBy("source_date")
    .saveAsTable(TARGET_TABLE)
)


# ============================================================
# 6. POST-WRITE VALIDATION
# ============================================================

count = (
    spark.table(TARGET_TABLE)
    .filter(col("source_date") == TARGET_DATE)
    .count()
)

print(
    f"✅ gold_brand_performance complete: "
    f"{count:,} rows for {TARGET_DATE}"
)

if count == 0:
    raise RuntimeError(
        "❌ gold_brand_performance write succeeded "
        "but 0 rows found"
    )


# ============================================================
# 7. FINAL VALIDATION
# ============================================================

print("=" * 70)
print("✅ GOLD BRAND PERFORMANCE SUCCESS")
print("=" * 70)

GOLD BRAND PERFORMANCE
Source table : billing.sales_data.billing_silver
Target table : billing.sales_data.gold_brand_performance
Target date  : 2026-09-18
Excluded     : ['Ho Marlboro']
📥 Silver rows for 2026-09-18: 47,051
📊 Rows after store exclusion: 47,018
📊 Brand aggregate rows: 1,198
🗑️ Cleared existing gold_brand_performance for 2026-09-18
✅ gold_brand_performance complete: 1,198 rows for 2026-09-18
✅ GOLD BRAND PERFORMANCE SUCCESS
